In [1]:
include("../src/naml.jl")

frechet_mean

In [2]:
p = 3
prec = 20
K = PadicField(p, prec)

Field of 3-adic numbers

In [3]:
# For testing. Creates the DAG matrix representation
# of a line with `n` nodes. Index of top node is `1`, and we go upwards.
function mk_line_matrix(n::Int64) 
    D = Matrix{Bool}(undef, n, n)
    for i in 1:n 
        for j in 1:n 
            D[i, j] = (j < i)
        end 
    end
    return D
end     

D = mk_line_matrix(5)

5×5 Matrix{Bool}:
 0  0  0  0  0
 1  0  0  0  0
 1  1  0  0  0
 1  1  1  0  0
 1  1  1  1  0

Here we're trying to embed a DAG in the disc space. We represent DAGs using a boolean matrix $D$, where $D_{ij}$ is true iff there is an edge $j \to i$. 

The main loss function we work with is
$$
    \mathcal{L} = \sum_{D_{ij} = 1} \log \frac{e ^ {- d(\theta_i, \theta_j)}}{\sum_{D_{ik} = 0} e ^ {- d(\theta_i, \theta_k)}}
$$
where $\theta_i$ is the polydisc embedding corresponding to node $i$. 
This is basically the same as is used in this paper: https://arxiv.org/pdf/1705.08039 (but working with the polydisc space rather than the hyperbolic disc!)

In [ ]:
# TODO: implement various operations on abstract polydisc functions and re-implement this using these operations

# "Naive" loss function, using absolute polynomials to try and capture the distance between nodes. This does not work well!
function init_rational_loss(D::Matrix{Bool}, K)
    # TODO: do we also want some kind of method for sparse evaluation?
    # For now type is hard-coded. Should be changed later on...
    linear_polynomials = Matrix{LinearPolynomial{PadicFieldElem}}(undef, size(D, 1), size(D, 2)) #([], K(0))
    for i in axes(D, 1)
        for j in axes(D, 2)
            linear_polynomials[i, j] = LinearPolynomial([u == i ? K(1) : K(0) for u in axes(D, 1)] - [u == j ? K(1) : K(0) for u in axes(D, 1)], K(0))
        end
    end
    linear_polynomials = map(batch_evaluate_init, linear_polynomials) 

    function eval_matrix(params::ValuationPolydisc{S, T})::Matrix{Float64} where S where T
        return map(f -> f(params), linear_polynomials)
    end

    # This can be vectorised!
    function eval(params::ValuationPolydisc{S, T})::Float64 where S where T
        pre_computed = eval_matrix(params)
        result = Float64(0)
        for i in axes(D, 1)
            den = Float64(0)
            num = Float64(0)
            for j in axes(D, 2)
                if !D[i, j] 
                    den += pre_computed[i, j]
                else
                    num += pre_computed[i, j]
                end
            end
            result += num / den
        end
        return result
    end
    return Loss(p -> map(eval, p), p -> 0)
end

# TODO: we may want to add a term to penalise lack of injectivity in the embedding! But maybe this is just something to be solved by initialising in a
# smarter way?

# The loss function used in the Meta paper.
function init_distance_loss(D::Matrix{Bool}, K::Ring)
    # Quick and dirty implementation for now: we'll vectorise and optimise later.

    function eval(params::ValuationPolydisc{S, T})::Float64 where S where T
        result = Float64(0)
        # TODO: implement iterate for polydiscs
        discs = components(params)
        # TODO: debug issue with unique!(discs)
        # println("Duplicates: ", length([(i, j) for i in discs for j in discs if i == j]))
        result += length([(i, j) for i in discs for j in discs if i == j]) * 1000
        for i in axes(D, 1)
            den = Float64(0)
            num = Float64(0)
            for j in axes(D, 2)
                if !D[i, j] 
                    den += exp(- dist(discs[i], discs[j]))
                else
                    num += exp(-dist(discs[i], discs[j]))
                end
            end
            if num == 0
                continue
            else
                result += log(num / den)
            end
        end
        return result
    end
    return Loss(p -> map(eval, p), p -> 0)
end 

# The loss function used in the Meta paper.
function init_hyp_distance_loss(D::Matrix{Bool}, K::Ring)
    # Quick and dirty implementation for now: we'll vectorise and optimise later.

    function eval(params::ValuationPolydisc{S, T})::Float64 where S where T
        result = Float64(0)
        # TODO: implement iterate for polydiscs
        discs = components(params)
        for i in axes(D, 1)
            den = Float64(0)
            num = Float64(0)
            for j in axes(D, 2)
                if !D[i, j] 
                    den += exp(- hyp_dist(discs[i], discs[j]))
                else
                    num += exp(- hyp_dist(discs[i], discs[j]))
                end
            end
            if num == 0
                continue
            else
                result += log(num / den)
            end
        end
        return result
    end
    return Loss(p -> map(eval, p), p -> 0)
end 

## TODO: generic Gauss point constructor
function mk_initial_for(D::Matrix{Bool})
    return ValuationPolydisc(zeros(K, size(D, 1)), zeros(Int64, size(D, 1)))
end

ℓ_poly = init_rational_loss(D, K)
ℓ_dist = init_distance_loss(D, K)
ℓ_hyp_dist = init_hyp_distance_loss(D, K)

next_branch = 1

# Here we don't want to force every coordinate to shrink! 
# The whole point being that which coordinates need to shrink more should be decided by the loss in this situation. 
config = (false, 1)

initial = mk_initial_for(D)

# We optimise using Greedy descent
greedy_optim_with_poly::OptimSetup = greedy_descent_init(initial, ℓ_poly, next_branch, config)

Base.Meta.ParseError: ParseError:
# Error @ /home/paul/Documents/Dev/naml/test/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W4sZmlsZQ==.jl:47:52

    function eval(params::ValuationPolydisc{S, T}) -> Float64 where S where T
#                                                  └┘ ── invalid identifier

In [5]:
N_epochs = 1000
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_with_poly))
    step!(greedy_optim_with_poly)
end 
t2 = time()

println("Result of optimisation process is $(greedy_optim_with_poly.param)")

Loss at epoch 1 is 2.867971989969915e-9
Loss at epoch 2 is 2.867971989969915e-9
Loss at epoch 3 is 2.867971989969915e-9
Loss at epoch 4 is 2.867971989969915e-9
Loss at epoch 5 is 2.867971989969915e-9
Loss at epoch 6 is 2.867971989969915e-9
Loss at epoch 7 is 2.867971989969915e-9
Loss at epoch 8 is 2.867971989969915e-9
Loss at epoch 9 is 2.867971989969915e-9
Loss at epoch 10 is 2.867971989969915e-9
Loss at epoch 11 is 2.867971989969915e-9
Loss at epoch 12 is 2.867971989969915e-9
Loss at epoch 13 is 2.867971989969915e-9
Loss at epoch 14 is 2.867971989969915e-9
Loss at epoch 15 is 2.867971989969915e-9
Loss at epoch 16 is 2.867971989969915e-9
Loss at epoch 17 is 2.867971989969915e-9
Loss at epoch 18 is 2.867971989969915e-9
Loss at epoch 19 is 2.867971989969915e-9
Loss at epoch 20 is 2.867971989969915e-9
Loss at epoch 21 is 2.867971989969915e-9
Loss at epoch 22 is 2.867971989969915e-9
Loss at epoch 23 is 2.867971989969915e-9
Loss at epoch 24 is 2.867971989969915e-9
Loss at epoch 25 is 2.867

In [6]:
N_epochs = 1000
greedy_optim_with_distance::OptimSetup = greedy_descent_init(initial, ℓ_dist, next_branch, config)
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_with_distance))
    step!(greedy_optim_with_distance)
end 
t2 = time()

println("Result of optimisation process is $(greedy_optim_with_distance.param)")

Loss at epoch 1 is 25000.0
Loss at epoch 2 is 16998.747881921197
Loss at epoch 3 is 10997.783117030269
Loss at epoch 4 is 6997.162326730105
Loss at epoch 5 is 6996.725317091345
Loss at epoch 6 is 4996.948904272691
Loss at epoch 7 is 4996.622357128499
Loss at epoch 8 is 4996.4155326430855
Loss at epoch 9 is 4996.240281988336
Loss at epoch 10 is 4996.076664189848
Loss at epoch 11 is 4995.964570525032
Loss at epoch 12 is 4995.891206436206
Loss at epoch 13 is 4995.805603389038
Loss at epoch 14 is 4995.726299996532
Loss at epoch 15 is 4995.665469825694
Loss at epoch 16 is 4995.608813858551
Loss at epoch 17 is 4995.571383517964
Loss at epoch 18 is 4995.539394172905
Loss at epoch 19 is 4995.50910015857
Loss at epoch 20 is 4995.488347106183
Loss at epoch 21 is 4995.469421048095
Loss at epoch 22 is 4995.456860562484
Loss at epoch 23 is 4995.4455118422675
Loss at epoch 24 is 4995.435334784887
Loss at epoch 25 is 4995.42836656916
Loss at epoch 26 is 4995.422053355832
Loss at epoch 27 is 4995.4178

In [7]:
N_epochs = 1000
greedy_optim_with_hyp_distance::OptimSetup = greedy_descent_init(initial, ℓ_hyp_dist, next_branch, config)
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_with_hyp_distance))
    step!(greedy_optim_with_hyp_distance)
end 
t2 = time()

println("Result of optimisation process is $(greedy_optim_with_hyp_distance.param)")

Loss at epoch 1 is 0.0
Loss at epoch 2 is -0.4303209517310074
Loss at epoch 3 is -1.5291719008300424
Loss at epoch 4 is -3.1046627918662137
Loss at epoch 5 is -4.929562847023989
Loss at epoch 6 is -6.862059673222248
Loss at epoch 7 is -8.836775855065564
Loss at epoch 8 is -10.827411649826512
Loss at epoch 9 is -12.823958158125954
Loss at epoch 10 is -14.822686521889263
Loss at epoch 11 is -16.822218554803676
Loss at epoch 12 is -18.822046377904226
Loss at epoch 13 is -20.821983034661915
Loss at epoch 14 is -22.82195973159273
Loss at epoch 15 is -24.821951158819534
Loss at epoch 16 is -26.82194800506532
Loss at epoch 17 is -28.82194684486301
Loss at epoch 18 is -30.821946418048302
Loss at epoch 19 is -32.821946261031925
Loss at epoch 20 is -34.82194620326884
Loss at epoch 21 is -36.821946182018976
Loss at epoch 22 is -40.821946174201585
Loss at epoch 23 is -44.821946171325735
Loss at epoch 24 is -48.821946170267765
Loss at epoch 25 is -52.82194616987856
Loss at epoch 26 is -56.821946169